# Level 3 - Localization

Predict the polygon bounding the locally manipulated region of each test image. The
metric is mean IoU over the ground-truth edited images only, which is measured here
rather than read off the overview. See section 5.

What was checked before writing this:

1. The format is polygons, not RLE. The overview contradicts itself, with the
   submission section saying normalized polygon coordinates and the evaluation section
   saying RLE. train/segmentation.csv settles it: normalized coordinates at 6 decimals,
   space separated, in `x y x y ...` order.
2. Coordinate order is `x y`. Checked two ways. The SRM residual is 1.38x stronger
   inside the mask than outside under `x y` but sits at chance (1.02) under `y x`, and
   rendering the smallest masks shows `x y` contours wrapping inserted people and a
   head in a car window while `y x` lands on empty sky.
3. `;` separates connected components, not holes. Of 416 secondary parts in val, none
   have their centroid inside the largest part. So decoding is a union of filled
   polygons, and ground-truth masks are hole-free by construction: hole-filling changes
   0 of 326 val masks. Predictions get hole-filled to match.
4. The grid is 640. Every coordinate is k/640 for integer k in [0, 639], exact to
   6-decimal truncation. Consecutive contour points step by (1,1) more often than (0,1)
   or (1,0), which is the cv2.findContours signature, so the ground truth was traced
   off a binary mask.
5. The codec is pixel-lossless. mask -> findContours -> polygon -> fillPoly round-trips
   at IoU 1.000000 on all 326 val masks, minimum and not just mean. The representation
   costs nothing, so every lost point is model error.
6. The image set is byte-identical to Levels 1 and 2. Same 7857/978/993 image_ids, md5
   matches on sampled train/val/test files. Locally the images are symlinked to Level
   2's copy rather than downloaded again.
7. has_polygon is exactly classe == 2, with three exceptions: three train images are
   labelled Cat 2 but carry a blank polygon. Those are dropped, since they would teach
   the model that an edit can have an empty mask.

## There is no free floor on the score

The overview says an empty ground truth with an empty prediction scores 1.0, which
would hand an all-blank submission 662/993 = 0.6667. Submitting the unmodified
sample_submission.csv scored exactly 0.00000, so unedited images are not scored at all:

```
score = mean IoU over the ~331 ground-truth Cat-2 test images
```

On the 326 val edit masks the reference points are the scores themselves: full image
0.196, best fixed "average" mask 0.344, perfect bounding box 0.564, perfect masks 1.0.
The median mask fills only 0.59 of its bounding box because the edits are object
shaped, so anything box shaped is capped at 0.564 and real masks are required.

## Measured so far

| | OOF mIoU | Public |
|---|---|---|
| v2 holdout, 1 model | 0.8131 | 0.79104 |
| v3 5-fold cv, 5-model ensemble | 0.8096 | 0.84648 |
| this notebook, effnetv2_l 5-fold + TTA | 0.8417 | |
| 3-run weighted blend, the submitted entry | 0.8500 | 0.89534 |

Ensembling is the lever that moved the score. Going from one model to a 5-fold ensemble
was worth +0.0554 public, far more than the out-of-fold numbers suggested, and the OOF
cannot see it because no OOF image can be scored by the ensemble without leakage.
Blending across encoders and seeds on top of that took OOF from 0.8417 to 0.8500.

Post-processing is exhausted by comparison: an oracle per-image threshold is worth only
+0.013, dilation only hurts, and D4 flip TTA measures +0.004 +- 0.003.

This notebook is the effnetv2_l run, the strongest single model of the set and the one
carrying weight 0.4 in the final blend.

The failure mode is small regions. The IoU < 0.10 images from v3 have median
ground-truth area 0.0505 against 0.1495 overall, while the model predicts 0.1672 for
them. That kind of over-covering is what a Dice-dominated objective does when large
regions supply most of the gradient, which is what lovasz_weight is for.

## Which images get a polygon

Deciding blank against not-blank is Level 2's task, and Level 2 solved it: its 5-fold
OOF Cat-2 gate makes 3 false positives and 7 false negatives in 8835 images, so about
one error is expected over 993 test images. This notebook does not try to beat that. It
reads Level 2's predictions.npz, mounted through kernel_sources, and segments only the
331 of 993 images Level 2 calls Cat 2. Training therefore uses Cat-2 images only, so
none of the capacity goes on the 2/3 of the data the gate already handles.

One consequence worth stating: for a gated image an empty prediction scores a
guaranteed 0 while any overlapping prediction scores more, so post-processing never
emits an empty mask for a gated image.

## Carried over from Levels 1 and 2

- Prior-initialise the head. A fresh head at 640px emitted logits near +-8 and Level 1
  spent its first steps collapsing a loss of 8.07. Zero weight plus bias = logit(prior)
  moved a short trial run from AUC 0.4269 to 0.8231. Here the prior is the mean mask area
  fraction, around 0.19, so the model starts out predicting the average edit size.
- use_srm = True. On Level 2 the SRM residual cut missed edits from 11 to 6, the best
  single change measured there. Localizing a local edit is the same signal spatially
  resolved, so it is on by default rather than treated as an experiment.
- No resampling augmentation. D4 only, rot90 and flip, which permutes pixels without
  interpolation so the forensic traces survive. Colour jitter, blur and scaling would
  destroy the evidence being localised.
- Gate the LR scheduler on the optimiser actually stepping. GradScaler skips the first
  step at initial scale 65536, and stepping the schedule anyway is wrong.
- The OOF noise floor is real. cudnn.benchmark plus AMP move Level 1's error count by
  around +-2 between identical runs. Here the metric is a mean of continuous per-image
  IoUs, which behaves far better: the val mIoU standard error is ~0.011 over 326
  images, so a single run is readable on this level in a way it never was on 1 and 2.
- Dump raw probabilities. Every good Level 1 and 2 result came from re-deciding and
  blending predictions.npz locally on CPU. Test masks go out at full 640 resolution so
  a submission can be rebuilt without any GPU time.

## Config

In [ ]:
class CFG:
    seed = 42
    n_folds = 5

    # model
    # Encoder has to be a CNN. The decoder consumes multi-scale features and the net
    # runs at full 640 resolution, which fixed-position-embedding ViTs reject.
    model_name = "tf_efficientnetv2_l.in21k_ft_in1k"
    pretrained = True
    drop_path_rate = 0.2
    dec_channels = (256, 128, 64, 32)
    # Refine at full 640 instead of bilinear-upsampling logits from the decoder's
    # stride-2 output. See the note in SegNet.
    refine_full_res = False
    refine_channels = 32

    # forensic input channels
    # 6 channels: RGB + fixed SRM high-pass residual, Level 2's best single change
    # (missed edits 11 -> 6).
    use_srm = True
    srm_gain = 10.0

    # resolution
    train_size = 640       # native; no resizing at evaluation
    eval_size = 640
    # Random scale jitter of +-this fraction on the training crop, applied to image
    # and mask together. 0.0 reproduces the earlier D4-only recipe. See the note in
    # SegDataset for why the Levels 1/2 ban on resampling may not carry over to a
    # localisation task.
    scale_aug = 0.0

    # loss
    # Dice optimises overlap directly, which is what IoU scores, while BCE keeps the
    # per-pixel gradient well behaved early on. Equal weight is the usual default
    # and worth sweeping later.
    bce_weight = 0.5
    dice_weight = 0.5
    # Lovasz-hinge is a convex surrogate for the Jaccard index itself, so it
    # optimises the scored quantity directly rather than Dice's proxy, and it is much
    # less biased toward large regions. 0.0 keeps the earlier objective; raise it to
    # trade against dice_weight. See the note on lovasz_hinge.
    lovasz_weight = 0.0

    # optimisation
    # The decoder is randomly initialised, so this needs more epochs than Level 2's
    # 3. batch 4 x accum 8 = effective 32, matching Levels 1/2. Batch 4 rather than
    # 8 because the decoder's full-resolution activations are what fill the T4.
    epochs = 16
    batch_size = 2
    grad_accum = 16
    eval_batch_size = 8
    lr = 3e-4
    weight_decay = 1e-2
    warmup_frac = 0.1
    max_grad_norm = 1.0
    num_workers = 4
    amp = True

    # Fraction of Cat-0/Cat-1 images (all-zero masks) to mix into training, relative
    # to the number of Cat-2 images. 0.0 by default, since the Level 2 gate means the
    # model is never asked about a non-edit and negatives would spend capacity on a
    # decision already made.
    neg_ratio = 0.0

    # inference
    # D4 flip TTA, averaged in probability space after un-flipping. Off by default
    # since Levels 1/2 both measured TTA as slightly harmful for classification, but
    # boundary averaging is a different mechanism, so measure it before believing.
    tta = True

    # post-processing, swept on OOF rather than guessed
    thresholds = (0.3, 0.4, 0.5, 0.6, 0.7)
    min_area_fracs = (0.0, 0.01, 0.03)   # drop components below this fraction of the largest
    close_pxs = (0, 5)                   # morphological closing radius

    # output
    out_dir = "/kaggle/working"
    dump_oof_size = 320   # OOF probability maps dumped at this size (uint8)
    dump_test_full = True # dump test maps at full 640 so submissions rebuild on CPU

## Imports

In [ ]:
import gc
import os
import random
import time
from glob import glob

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

# cv2 spawns its own thread pool which fights the DataLoader workers.
cv2.setNumThreads(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = CFG.amp and DEVICE == "cuda"


def autocast():
    return torch.amp.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG.seed)
torch.backends.cudnn.benchmark = True

os.makedirs(CFG.out_dir, exist_ok=True)


## Data paths

The annotation file here is segmentation.csv, with columns image_id and polygon, not
the ground_truth.csv of Levels 1 and 2.

In [ ]:
GRID = 640  # the coordinate grid the ground truth is quantised to; == image size


DATA_ROOT = "/kaggle/input/gensivana-real-or-fake-level-3-localization"

TEST_DIR = os.path.join(DATA_ROOT, "test", "images")


def load_segmentation(split: str) -> pd.DataFrame:
    """Load a segmentation.csv. Adds `dir` and `has_mask`; keeps `polygon` verbatim."""
    df = pd.read_csv(os.path.join(DATA_ROOT, split, "segmentation.csv"),
                     keep_default_na=False, dtype=str)
    poly_col = next((c for c in ("polygon", "polygone", "segmentation") if c in df.columns), None)
    if poly_col is None:
        raise KeyError(f"No polygon column in {split}/segmentation.csv: {list(df.columns)}")
    df = df.rename(columns={poly_col: "polygon"})
    df["dir"] = os.path.join(DATA_ROOT, split, "images")
    # A blank polygon is a single space in the provided files, so treat any
    # whitespace-only value as "no manipulated region".
    df["has_mask"] = df.polygon.str.strip().str.len() > 0
    return df[["image_id", "polygon", "dir", "has_mask"]]


train_df = load_segmentation("train")
val_df = load_segmentation("val")

sample_sub = pd.read_csv(os.path.join(DATA_ROOT, "sample_submission.csv"),
                         keep_default_na=False, dtype=str)
test_ids = sample_sub["image_id"].tolist()
test_df = pd.DataFrame({"image_id": test_ids, "dir": TEST_DIR})

for name, df in (("train", train_df), ("val", val_df)):
    n = len(df)
    k = int(df.has_mask.sum())
    print(f"{name:5} {n:>5}   with mask {k:>5} ({k / n:.3f})   blank {n - k:>5}")
print(f"test  {len(test_df):>5}")

# Fail loudly now rather than after an hour of training.
assert train_df.image_id.is_unique and val_df.image_id.is_unique
assert set(train_df.image_id) & set(val_df.image_id) == set(), "train/val leak"
assert len(set(test_ids)) == len(test_ids), "duplicate ids in sample_submission"

## Polygon / mask codec

The ground truth was produced by tracing a binary mask with cv2.findContours, going by
the (1,1) step signature, so encoding the same way keeps our polygons stylistically
indistinguishable from it and whatever rasteriser the grader uses treats both alike.
RETR_EXTERNAL also means ground-truth masks are hole-free, so mask_to_polygon fills
holes implicitly and postprocess fills them explicitly.

The self-test below is the load-bearing claim of the notebook: it asserts the round
trip is exactly lossless, so all remaining error is model error.

In [ ]:
def parse_polygon(text: str) -> list[np.ndarray]:
    """'x y x y ; x y ...' -> list of (n, 2) float arrays of normalized x, y."""
    parts = []
    for chunk in str(text).split(";"):
        toks = chunk.split()
        if len(toks) < 6:            # fewer than 3 points cannot enclose area
            continue
        if len(toks) % 2:
            raise ValueError(f"odd coordinate count ({len(toks)}) in polygon chunk")
        parts.append(np.asarray(toks, dtype=np.float64).reshape(-1, 2))
    return parts


def polygon_to_mask(text: str, size: int = GRID) -> np.ndarray:
    """Decode to a uint8 {0,1} mask. Blank/whitespace -> all zeros."""
    mask = np.zeros((size, size), dtype=np.uint8)
    for pts in parse_polygon(text):
        cv2.fillPoly(mask, [np.round(pts * size).astype(np.int32)], 1)
    return mask


def mask_to_polygon(mask: np.ndarray) -> str:
    """Encode a binary mask as the ground truth encodes it. Empty mask -> ' '."""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_NONE)
    size = mask.shape[0]
    parts = []
    for c in contours:
        pts = c.reshape(-1, 2)
        if len(pts) < 3:
            continue
        parts.append(" ".join(f"{x / size:.6f} {y / size:.6f}" for x, y in pts))
    return ";".join(parts) if parts else " "


def iou(a: np.ndarray, b: np.ndarray) -> float:
    """IoU of two boolean masks. Both empty -> 1.0, matching the stated metric."""
    union = np.count_nonzero(a | b)
    if union == 0:
        return 1.0
    return np.count_nonzero(a & b) / union


# Self-test: decode -> encode -> decode must be bit-identical, on real annotations.
_probe = pd.concat([train_df[train_df.has_mask].head(60),
                    val_df[val_df.has_mask].head(60)])
_worst = 1.0
for _text in _probe.polygon:
    _m = polygon_to_mask(_text)
    _m2 = polygon_to_mask(mask_to_polygon(_m))
    _worst = min(_worst, iou(_m.astype(bool), _m2.astype(bool)))
    assert np.array_equal(_m, _m2), "codec is not lossless, do not train on this"
print(f"codec self-test: {len(_probe)} annotations round-trip bit-identically "
      f"(worst IoU {_worst:.6f})")

# Three train images are labelled Cat 2 upstream but carry a blank polygon. Selecting
# on has_mask already excludes them; this just makes the count visible.
print(f"train rows with a mask: {int(train_df.has_mask.sum())} of {len(train_df)}")

## The Level 2 gate

Read from Level 2's committed predictions.npz, mounted through `kernel_sources`. Its
argmax decides which test images are local edits and therefore which get a polygon at
all. Level 3 never re-learns that decision.

In [ ]:
def load_cat2_gate(ids: list[str]) -> np.ndarray | None:
    """Boolean 'is a local edit' per test id, from Level 2's 3-class classifier."""
    # Recursive, because Kaggle mounts competition data under
    # /kaggle/input/competitions/<slug> and a kernel_sources output at its own nesting
    # depth, so a fixed number of path segments is not safe to assume. An earlier run
    # missed the gate for exactly this reason.
    candidates = sorted(glob("/kaggle/input/**/predictions.npz", recursive=True))
    candidates += sorted(glob("../artifacts/**/predictions.npz", recursive=True))
    candidates += sorted(glob("Level_2-Classification/**/predictions.npz", recursive=True))
    candidates += sorted(glob("../../Level_2-Classification/**/predictions.npz",
                             recursive=True))

    found = []
    for path in candidates:
        try:
            d = np.load(path, allow_pickle=True)
            if "test_probs" not in d or "test_ids" not in d:
                continue
            probs = d["test_probs"]
            if probs.ndim != 2 or probs.shape[1] != 3:
                continue
            order = {str(i): k for k, i in enumerate(d["test_ids"])}
            if not set(ids) <= set(order):
                continue
            found.append((path, probs[[order[i] for i in ids]]))
        except Exception as exc:                      # noqa: BLE001
            print(f"  skipping {path}: {exc}")

    if not found:
        return None
    # Several Level 2 runs may be mounted. Averaging them matches what the local
    # blend did, and they all agreed on the argmax anyway.
    for path, _ in found:
        print(f"  gate source: {path}")
    mean_probs = np.mean([p for _, p in found], axis=0)
    return mean_probs.argmax(axis=1) == 2


GATE = load_cat2_gate(test_ids)
assert GATE is not None, ("Level 2 predictions not found. Attach "
                          "gensivana-l2-classification through kernel_sources.")

n_edit = int(GATE.sum())
print(f"Level 2 gate: {n_edit} of {len(test_ids)} test images are local edits "
      f"({n_edit / len(test_ids):.3f}); {len(test_ids) - n_edit} take a blank polygon")

# The blanks earn nothing, since non-edit images are not scored at all (see
# projected_score). The gate's only remaining job is to avoid blanking a true edit,
# which costs a full 1.0. Its false-negative rate is 7 in 2942 edits, so it should
# cost ~0.8 of the ~331 test edits, around -0.002 of score.
#
# Emitting a polygon everywhere would recover that at no measured cost, since false
# positives are not scored. Not doing it anyway: the gain is ~+0.002, but if the
# private scorer did turn out to average over all 993 images, blanket polygons would
# cost ~0.67. The gate is the cheap hedge against public and private differing.
print(f"gate false negatives cost ~{7 / 2942 * n_edit:.1f} images "
      f"(~{7 / 2942:.4f} of score); kept as a hedge, see the comment above")


def projected_score(mean_iou_on_edits: float) -> float:
    """The leaderboard score is the mean IoU over ground-truth edit images.

    Measured, not assumed. The overview documents an empty-mask edge case, "ground
    truth empty AND prediction empty -> IoU = 1.0", which would make an all-blank
    submission score 662/993 = 0.6667. Submitting the unmodified sample_submission
    scored exactly 0.00000, so images with no ground-truth polygon are simply not
    scored and that documented edge case is wrong the same way the file's RLE claim
    is wrong.

    Two consequences. The 2/3 of the test set that is not edited contributes nothing,
    so there is no free floor and every point comes from mask quality. And a polygon
    on a Cat-0/Cat-1 image is never scored, so a false positive costs nothing while a
    blank on a true edit costs a full 1.0.
    """
    return mean_iou_on_edits

## Baselines

Printed every run so a mediocre mIoU cannot look impressive. Perfect bounding box is
the number that matters most, since any method predicting boxes rather than masks is
capped there.

In [ ]:
_ref = val_df[val_df.has_mask]
_ref_masks = [polygon_to_mask(t).astype(bool) for t in _ref.polygon.head(200)]
_full = np.ones((GRID, GRID), dtype=bool)

_areas = np.array([m.mean() for m in _ref_masks])
print(f"reference set: {len(_ref_masks)} val edit masks")
print(f"  mask area fraction: median {np.median(_areas):.4f}  "
      f"p5 {np.percentile(_areas, 5):.4f}  p95 {np.percentile(_areas, 95):.4f}")

_bbox = []
for _m in _ref_masks:
    ys, xs = np.nonzero(_m)
    _b = np.zeros_like(_m)
    _b[ys.min():ys.max() + 1, xs.min():xs.max() + 1] = True
    _bbox.append(iou(_m, _b))

for _name, _v in [
    ("all-blank (measured: 0.00000)", 0.0),
    ("predict the full image", float(np.mean([iou(m, _full) for m in _ref_masks]))),
    ("perfect bounding box", float(np.mean(_bbox))),
    ("perfect masks", 1.0),
]:
    print(f"  {_name:32} mIoU {_v:.4f}   -> score {projected_score(_v):.4f}")
print("The score is this mIoU; there is no free floor from the unedited 2/3.")
print("Reference: v2 (holdout, 8 epochs) scored 0.79104 public from OOF 0.8131.")

## Dataset

Masks are rasterised on the fly from the polygon string, a few hundred microseconds
each, which avoids writing 3000 mask PNGs. Augmentation is D4 applied to image and mask
together: it permutes pixels without interpolation, so the resampling traces that
reveal an edit survive.

In [ ]:
class SegDataset(Dataset):
    def __init__(self, df: pd.DataFrame, size: int, train: bool):
        self.ids = df.image_id.tolist()
        self.dirs = df.dir.tolist()
        self.polys = df.polygon.tolist() if "polygon" in df.columns else None
        self.size = size
        self.train = train

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, idx: int):
        path = os.path.join(self.dirs[idx], self.ids[idx])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"failed to read {path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = None
        if self.polys is not None:
            mask = polygon_to_mask(self.polys[idx], img.shape[0])

        h, w = img.shape[:2]

        # Scale/shift augmentation. Levels 1 and 2 banned resampling because they
        # detect forensic traces that interpolation destroys, but this is a
        # localisation task and the model clearly works semantically, with contours
        # wrapping inserted people. At 2942 training images against 52M parameters
        # the binding constraint is overfitting instead: val mIoU peaked at epoch 10
        # and fell for four more while train loss kept dropping. So the inherited ban
        # gets tested here rather than assumed.
        if self.train and getattr(CFG, "scale_aug", 0.0) > 0 and mask is not None:
            s = 1.0 + random.uniform(-CFG.scale_aug, CFG.scale_aug)
            side = int(round(self.size / s))
            side = max(64, min(side, min(h, w)))
            y = random.randint(0, h - side)
            x = random.randint(0, w - side)
            img = img[y:y + side, x:x + side]
            mask = mask[y:y + side, x:x + side]
            if side != self.size:
                # INTER_AREA when shrinking, INTER_LINEAR when enlarging. The mask
                # goes through NEAREST so it stays strictly binary.
                interp = cv2.INTER_AREA if side > self.size else cv2.INTER_LINEAR
                img = cv2.resize(img, (self.size, self.size), interpolation=interp)
                mask = cv2.resize(mask, (self.size, self.size),
                                  interpolation=cv2.INTER_NEAREST)
            h, w = img.shape[:2]

        if self.size < min(h, w):
            if self.train:
                y = random.randint(0, h - self.size)
                x = random.randint(0, w - self.size)
            else:
                y, x = (h - self.size) // 2, (w - self.size) // 2
            img = img[y:y + self.size, x:x + self.size]
            if mask is not None:
                mask = mask[y:y + self.size, x:x + self.size]

        if self.train:
            k = random.randint(0, 3)
            if k:
                img = np.rot90(img, k)
                if mask is not None:
                    mask = np.rot90(mask, k)
            if random.random() < 0.5:
                img = img[:, ::-1]
                if mask is not None:
                    mask = mask[:, ::-1]

        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        x_t = torch.from_numpy(img).float().div_(255.0)
        if mask is None:
            return x_t
        m_t = torch.from_numpy(np.ascontiguousarray(mask)).float().unsqueeze(0)
        return x_t, m_t


def make_loader(df: pd.DataFrame, size: int, train: bool, batch_size: int) -> DataLoader:
    return DataLoader(
        SegDataset(df, size, train),
        batch_size=batch_size,
        shuffle=train,
        drop_last=train,
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=CFG.num_workers > 0,
    )

## Model: timm encoder plus U-Net decoder

The decoder is written out rather than pulled from segmentation_models_pytorch on
purpose. This has to run reproducibly on Kaggle, and a pip install of a package that is
not in the image is one more thing that can break a committed run. features_only=True
gives the five encoder scales, and channel counts come from feature_info so swapping
the encoder needs no other edit.

In [ ]:
SRM_KERNEL = torch.tensor(
    [[-1.0, 2.0, -1.0],
     [2.0, -4.0, 2.0],
     [-1.0, 2.0, -1.0]]
) / 4.0

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class ConvBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetDecoder(nn.Module):
    """Standard U-Net up-path. Upsamples to each skip's exact size, so odd
    feature-map dimensions cannot silently misalign."""

    def __init__(self, enc_channels: list[int], dec_channels=(256, 128, 64, 32)):
        super().__init__()
        skips = enc_channels[-2::-1]              # deep -> shallow, excluding the last
        if len(dec_channels) != len(skips):
            raise ValueError(
                f"decoder needs {len(skips)} stages for encoder channels "
                f"{enc_channels}, got {len(dec_channels)}"
            )
        self.blocks = nn.ModuleList()
        c_in = enc_channels[-1]
        for skip_c, c_out in zip(skips, dec_channels):
            self.blocks.append(ConvBlock(c_in + skip_c, c_out))
            c_in = c_out
        self.out_channels = c_in

    def forward(self, feats: list[torch.Tensor]) -> torch.Tensor:
        x = feats[-1]
        for block, skip in zip(self.blocks, feats[-2::-1]):
            x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
            x = block(torch.cat([x, skip], dim=1))
        return x


class SegNet(nn.Module):
    """Owns preprocessing so training and inference cannot drift apart."""

    def __init__(self, cfg=CFG, pos_rate: float | None = None):
        super().__init__()
        self.use_srm = cfg.use_srm
        self.srm_gain = cfg.srm_gain
        in_chans = 6 if self.use_srm else 3

        self.encoder = timm.create_model(
            cfg.model_name,
            pretrained=cfg.pretrained,
            in_chans=in_chans,
            features_only=True,
            drop_path_rate=cfg.drop_path_rate,
        )
        enc_channels = list(self.encoder.feature_info.channels())
        self.decoder = UNetDecoder(enc_channels, cfg.dec_channels)

        # The encoder's shallowest feature is at stride 2, so the decoder output is
        # 320x320 and the logits used to be bilinear-upsampled to 640, meaning
        # boundaries were predicted at half resolution and interpolated. Ground truth
        # is pixel-precise at 640 and a 1 px boundary error already costs ~0.02 of
        # IoU, so this stage refines at full resolution with the raw input as the
        # skip, there being no encoder feature at stride 1.
        self.refine = None
        head_in = self.decoder.out_channels
        if getattr(cfg, "refine_full_res", False):
            self.refine = ConvBlock(self.decoder.out_channels + in_chans,
                                    cfg.refine_channels)
            head_in = cfg.refine_channels
        self.head = nn.Conv2d(head_in, 1, kernel_size=1)

        # Same prior-initialisation that fixed Level 1's loss-8.07 start. Zero weight
        # and bias = logit(mean mask area) makes the model start out predicting the
        # average edit size everywhere instead of saturated noise.
        nn.init.zeros_(self.head.weight)
        if pos_rate is not None:
            p = float(np.clip(pos_rate, 1e-4, 1 - 1e-4))
            nn.init.constant_(self.head.bias, float(np.log(p / (1 - p))))
        else:
            nn.init.zeros_(self.head.bias)

        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        self.register_buffer("srm", SRM_KERNEL.view(1, 1, 3, 3).repeat(3, 1, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: raw RGB in [0, 1], (B, 3, H, W). Returns logits at (B, 1, H, W)."""
        h, w = x.shape[-2:]
        rgb = (x - self.mean) / self.std
        if self.use_srm:
            # High-pass the raw pixels before normalisation, otherwise per-channel
            # scaling gets folded into the residual.
            residual = F.conv2d(x, self.srm, padding=1, groups=3) * self.srm_gain
            rgb = torch.cat([rgb, residual], dim=1)
        decoded = self.decoder(self.encoder(rgb))
        if self.refine is not None:
            # Upsample features, not logits, then refine at full resolution.
            decoded = F.interpolate(decoded, size=(h, w), mode="bilinear",
                                    align_corners=False)
            return self.head(self.refine(torch.cat([decoded, rgb], dim=1)))
        logits = self.head(decoded)
        return F.interpolate(logits, size=(h, w), mode="bilinear", align_corners=False)


def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor,
                   eps: float = 1.0) -> torch.Tensor:
    """1 - Dice, per image then averaged. Optimises overlap, which is what IoU scores."""
    probs = torch.sigmoid(logits).flatten(1)
    tgt = target.flatten(1)
    inter = (probs * tgt).sum(1)
    denom = probs.sum(1) + tgt.sum(1)
    return (1.0 - (2.0 * inter + eps) / (denom + eps)).mean()


def lovasz_grad(gt_sorted: torch.Tensor) -> torch.Tensor:
    """Gradient of the Lovasz extension of the Jaccard loss (Berman et al., 2018)."""
    p = gt_sorted.numel()
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.cumsum(0)
    union = gts + (1 - gt_sorted).cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:p - 1]
    return jaccard


def lovasz_hinge(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Per-image Lovasz hinge, averaged over the batch.

    Dice weights every pixel of an image equally, so a large region supplies most of
    the gradient and small ones are effectively ignored. That is the failure mode
    measured here: the IoU < 0.10 images have median ground-truth area 0.0505 against
    0.1495 overall, and the model over-covers them at 0.1672. Lovasz is a convex
    surrogate for the Jaccard index itself, so it optimises the scored quantity
    directly and is much less size-biased.

    Computed per image and in fp32, since the sort is over 640*640 elements and the
    cumulative sums lose too much precision in fp16.
    """
    losses = []
    for logit, tgt in zip(logits, target):
        lab = tgt.reshape(-1).float()
        log = logit.reshape(-1).float()
        if lab.sum() == 0:                      # no positive pixels -> undefined
            losses.append(log.sum() * 0.0)
            continue
        signs = 2.0 * lab - 1.0
        errors = 1.0 - log * signs
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        losses.append(torch.dot(F.relu(errors_sorted), lovasz_grad(lab[perm])))
    return torch.stack(losses).mean()


class SegLoss(nn.Module):
    def __init__(self, bce_weight: float, dice_weight: float,
                 lovasz_weight: float = 0.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_w = bce_weight
        self.dice_w = dice_weight
        self.lovasz_w = lovasz_weight

    def forward(self, logits, target):
        loss = self.bce_w * self.bce(logits, target)
        if self.dice_w:
            loss = loss + self.dice_w * soft_dice_loss(logits, target)
        if self.lovasz_w:
            # Outside autocast, since the sort and cumsum need fp32 to stay stable.
            with torch.amp.autocast("cuda", enabled=False):
                loss = loss + self.lovasz_w * lovasz_hinge(logits.float(), target)
        return loss

## Post-processing and scoring

Two facts drive this. Ground-truth masks are hole-free by construction, so holes get
filled. And for a gated image an empty prediction scores a guaranteed 0 while any
overlap scores more, so postprocess never returns empty when asked not to: it falls
back to the single most confident blob.

In [ ]:
def postprocess(prob: np.ndarray, threshold: float, min_area_frac: float = 0.0,
                close_px: int = 0, allow_empty: bool = False) -> np.ndarray:
    """Probability map -> clean binary mask, hole-filled and (optionally) non-empty."""
    mask = (prob >= threshold).astype(np.uint8)

    if close_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close_px + 1,) * 2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

    if not mask.any() and not allow_empty:
        # Never hand a gated image an empty mask. IoU 0 is guaranteed, while any
        # overlap is worth something, so take the most confident region instead.
        mask = (prob >= max(prob.max() * 0.5, 1e-6)).astype(np.uint8)
        if not mask.any():
            mask = (prob >= prob.max()).astype(np.uint8)

    if min_area_frac > 0 and mask.any():
        n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        if n > 2:
            areas = stats[1:, cv2.CC_STAT_AREA]
            keep = 1 + np.flatnonzero(areas >= min_area_frac * areas.max())
            mask = np.isin(labels, keep).astype(np.uint8)

    # Fill holes: RETR_EXTERNAL + filled draw, matching how the ground truth was made.
    if mask.any():
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        filled = np.zeros_like(mask)
        cv2.drawContours(filled, contours, -1, 1, thickness=-1)
        mask = filled

    return mask


def mean_iou(probs: np.ndarray, gt_masks: list[np.ndarray], threshold: float,
             min_area_frac: float = 0.0, close_px: int = 0) -> float:
    """Mean IoU over a set of edit images, at full resolution."""
    scores = []
    for prob, gt in zip(probs, gt_masks):
        pred = postprocess(prob.astype(np.float32) / 255.0, threshold,
                           min_area_frac, close_px)
        scores.append(iou(pred.astype(bool), gt))
    return float(np.mean(scores))

## Train one fold

Probability maps come back as uint8 rather than float32. At 640x640 the OOF set for
5-fold CV would otherwise be 4.8 GB of float, and a 1/255 quantisation step is far
below the threshold granularity being swept.

In [ ]:
@torch.no_grad()
def predict_masks(model: nn.Module, loader: DataLoader, tta: bool = False) -> np.ndarray:
    """Return (N, H, W) uint8 probability maps. With tta, averages 4 D4 flip views."""
    model.eval()
    out = []
    for batch in loader:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        x = x.to(DEVICE, non_blocking=True)
        with autocast():
            p = torch.sigmoid(model(x).float())
            if tta:
                # Each view must be un-flipped before averaging, or the maps cancel.
                for dims in ([3], [2], [2, 3]):
                    q = torch.sigmoid(model(torch.flip(x, dims=dims)).float())
                    p = p + torch.flip(q, dims=dims)
                p = p / 4.0
        out.append((p.squeeze(1) * 255.0).round().clamp(0, 255).to(torch.uint8).cpu().numpy())
    return np.concatenate(out)


def train_one_fold(fold_train: pd.DataFrame, fold_valid: pd.DataFrame,
                   tag: str) -> dict:
    """Train one segmentation model. Returns val/test probability maps and metrics."""
    print(f"\n[{tag}] train {len(fold_train)}  valid {len(fold_valid)}")

    train_loader = make_loader(fold_train, CFG.train_size, True, CFG.batch_size)
    valid_loader = make_loader(fold_valid, CFG.eval_size, False, CFG.eval_batch_size)

    # Ground-truth masks for the validation fold, once, at full resolution.
    gt_masks = [polygon_to_mask(t, CFG.eval_size).astype(bool)
                for t in fold_valid.polygon]
    # Prior comes from the training masks, not the validation ones. Taking it from
    # gt_masks would leak the validation area distribution into the init.
    train_areas = [polygon_to_mask(t).mean() for t in fold_train.polygon]
    pos_rate = float(np.mean(train_areas)) if train_areas else 0.19
    print(f"  head prior: mean train mask area {pos_rate:.4f} "
          f"-> bias {np.log(pos_rate / (1 - pos_rate)):.3f}")

    model = SegNet(CFG, pos_rate=pos_rate).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr,
                                  weight_decay=CFG.weight_decay)

    steps_per_epoch = max(1, len(train_loader) // CFG.grad_accum)
    total_steps = steps_per_epoch * CFG.epochs

    warmup_steps = max(2, int(total_steps * CFG.warmup_frac))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=CFG.lr, total_steps=total_steps,
        pct_start=warmup_steps / total_steps, anneal_strategy="cos")
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    criterion = SegLoss(CFG.bce_weight, CFG.dice_weight, CFG.lovasz_weight)

    # Select on mean IoU at threshold 0.5, so the metric itself but at a fixed
    # threshold, which keeps the checkpoint choice separate from a fitted one. The
    # threshold gets swept afterwards on pooled OOF.
    best_score, best_state, best_epoch = -1.0, None, -1

    for epoch in range(CFG.epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running, seen, t0 = 0.0, 0, time.time()

        for step, (x, y) in enumerate(train_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast():
                loss = criterion(model(x), y)

            scaler.scale(loss / CFG.grad_accum).backward()

            if (step + 1) % CFG.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                # GradScaler skips optimizer.step() on inf/nan grads, which it does
                # on the first step at the initial scale of 65536. Stepping the LR
                # schedule for an update that never happened is wrong.
                stepped = scaler.get_scale() >= scale_before
                if stepped and scheduler.last_epoch < total_steps - 1:
                    scheduler.step()

            running += loss.item() * x.size(0)
            seen += x.size(0)

        val_probs = predict_masks(model, valid_loader, tta=False)
        score = mean_iou(val_probs, gt_masks, 0.5)
        print(f"  ep{epoch} done in {time.time() - t0:.0f}s | "
              f"loss {running / max(seen, 1):.4f} | mIoU@0.5 {score:.4f} "
              f"| projected {projected_score(score):.4f}")

        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
            print(f"  ep{epoch} new best mIoU {best_score:.4f}")

    model.load_state_dict(best_state)

    test_loader = make_loader(test_df, CFG.eval_size, False, CFG.eval_batch_size)
    val_probs = predict_masks(model, valid_loader, tta=CFG.tta)
    test_probs = predict_masks(model, test_loader, tta=CFG.tta)

    final = mean_iou(val_probs, gt_masks, 0.5)
    print(f"[{tag}] best epoch {best_epoch}, tta={CFG.tta}: mIoU@0.5 {final:.4f}")

    del model, best_state
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "tag": tag,
        "best_epoch": best_epoch,
        "val_ids": fold_valid.image_id.tolist(),
        "val_polygons": fold_valid.polygon.tolist(),
        "val_probs": val_probs,
        "test_probs": test_probs,
        "mean_iou_at_0.5": final,
    }

## Run

Training uses Cat-2 images only. The Level 2 gate means the model is never asked about
a non-edit, so negatives would spend capacity on a decision already made, and
CFG.neg_ratio mixes some back in if that turns out to be wrong. Folds are stratified on
mask-area quintile so no fold gets a skewed size distribution.

In [ ]:
edit_train = train_df[train_df.has_mask].reset_index(drop=True)
edit_val = val_df[val_df.has_mask].reset_index(drop=True)

if CFG.neg_ratio > 0:
    rng = np.random.default_rng(CFG.seed)
    blanks = train_df[~train_df.has_mask]
    take = min(len(blanks), int(round(CFG.neg_ratio * len(edit_train))))
    picked = blanks.iloc[rng.permutation(len(blanks))[:take]]
    edit_train = pd.concat([edit_train, picked], ignore_index=True)
    print(f"mixed in {take} blank-mask negatives (neg_ratio={CFG.neg_ratio})")

print(f"edit images: train {len(edit_train)}  val {len(edit_val)}")

results = []
pool = pd.concat([edit_train, edit_val], ignore_index=True)

# Stratify on mask-area quintile so the folds agree on the size mix, which is the
# dominant source of per-image IoU variance.
area = np.array([polygon_to_mask(t).mean() for t in pool.polygon])
bins = np.digitize(area, np.quantile(area, [0.2, 0.4, 0.6, 0.8]))
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)

for fold, (tr_idx, va_idx) in enumerate(skf.split(pool, bins)):
    results.append(train_one_fold(
        pool.iloc[tr_idx].reset_index(drop=True),
        pool.iloc[va_idx].reset_index(drop=True),
        f"fold{fold}",
    ))

print(f"\ntrained {len(results)} models")

## Pooled out-of-fold results and the post-processing sweep

Swept in two stages rather than as a full grid: threshold first, then morphology at the
winning threshold. A full grid over 5 x 3 x 2 configs at 640x640 costs minutes for a
decision the staged sweep reaches just as well.

In [ ]:
oof_ids = [i for r in results for i in r["val_ids"]]
oof_probs = np.concatenate([r["val_probs"] for r in results])
oof_gt = [polygon_to_mask(p, CFG.eval_size).astype(bool)
          for r in results for p in r["val_polygons"]]

# Each fold is an equally valid estimator; average in probability space. Summed in
# place rather than via np.mean over a list: at 640x640 five float32 copies of the
# test maps would be ~8 GB held at once.
_acc = np.zeros(results[0]["test_probs"].shape, dtype=np.float32)
for r in results:
    _acc += r["test_probs"]
_acc /= len(results)
test_probs = _acc.round().clip(0, 255).astype(np.uint8)
del _acc

print(f"OOF n = {len(oof_ids)} edit images")

by_threshold = {t: mean_iou(oof_probs, oof_gt, t) for t in CFG.thresholds}
for t, v in by_threshold.items():
    print(f"  threshold {t:.2f}  mIoU {v:.4f}   -> projected {projected_score(v):.4f}")
best_threshold = max(by_threshold, key=by_threshold.get)
print(f"best threshold: {best_threshold:.2f} (mIoU {by_threshold[best_threshold]:.4f})")

by_morph = {}
for maf in CFG.min_area_fracs:
    for cp in CFG.close_pxs:
        by_morph[(maf, cp)] = mean_iou(oof_probs, oof_gt, best_threshold, maf, cp)
for (maf, cp), v in sorted(by_morph.items(), key=lambda kv: -kv[1]):
    print(f"  min_area {maf:.2f}  close {cp:>2}px  mIoU {v:.4f}   "
          f"-> projected {projected_score(v):.4f}")
BEST_MIN_AREA, BEST_CLOSE = max(by_morph, key=by_morph.get)
OOF_MEAN_IOU = by_morph[(BEST_MIN_AREA, BEST_CLOSE)]

print(f"\nchosen post-processing: threshold {best_threshold:.2f}, "
      f"min_area_frac {BEST_MIN_AREA:.2f}, close {BEST_CLOSE}px")
print(f"OOF mean IoU on edits : {OOF_MEAN_IOU:.4f}")
print(f"projected leaderboard : {projected_score(OOF_MEAN_IOU):.4f}   "
      f"(all-blank floor {projected_score(0.0):.4f})")

# Per-image IoU distribution. The mean hides whether a few total misses or many
# mediocre masks are costing the points, and those need different fixes.
per_image = np.array([
    iou(postprocess(p.astype(np.float32) / 255.0, best_threshold,
                    BEST_MIN_AREA, BEST_CLOSE).astype(bool), g)
    for p, g in zip(oof_probs, oof_gt)
])
print("\nper-image IoU distribution on edits:")
for q in (5, 25, 50, 75, 95):
    print(f"  p{q:<3} {np.percentile(per_image, q):.4f}")
for lo in (0.1, 0.25, 0.5, 0.75, 0.9):
    n = int((per_image < lo).sum())
    # Divide by the OOF size, not the test size, since the score is a mean over the
    # images being summarised here. Dividing by len(test_ids) made this meaningless
    # whenever the OOF set was not 993 images: it understated the cost in holdout mode
    # (326 images) and overstated it ~3x in cv mode (2942).
    print(f"  IoU < {lo:.2f}: {n:>4} images ({n / len(per_image):.3f}) "
          f"-- costs {(1 - per_image[per_image < lo]).sum() / len(per_image):.4f} of mIoU")

## Build the submission

The polygon column carries a single space for images with no predicted manipulation. A
malformed file is rejected without consuming a daily attempt, but a well-formed wrong
one costs one of five, so validate everything the grader could reject including that
every coordinate is a normalized pair.

In [ ]:
BLANK = " "


def build_submission(ids: list[str], probs: np.ndarray, gate: np.ndarray,
                     threshold: float, min_area_frac: float, close_px: int) -> pd.DataFrame:
    """One polygon per gated image, a blank for the rest."""
    polys = []
    for k, image_id in enumerate(ids):
        if not gate[k]:
            polys.append(BLANK)
            continue
        mask = postprocess(probs[k].astype(np.float32) / 255.0, threshold,
                           min_area_frac, close_px, allow_empty=False)
        polys.append(mask_to_polygon(mask))
    return pd.DataFrame({"image_id": ids, "polygon": polys})


def validate_submission(sub: pd.DataFrame, reference: pd.DataFrame) -> None:
    """Raise on anything the grader would reject."""
    assert list(sub.columns) == ["image_id", "polygon"], f"bad columns: {list(sub.columns)}"
    assert len(sub) == len(reference), f"expected {len(reference)} rows, got {len(sub)}"
    assert sub.image_id.is_unique, "duplicate image_id"
    assert set(sub.image_id) == set(reference.image_id), "image_id set does not match test/"
    assert sub.notna().all().all(), "null values present"
    assert not sub.polygon.str.contains(",").any(), "comma in a polygon would break the CSV"

    n_blank = 0
    for image_id, text in zip(sub.image_id, sub.polygon):
        if not text.strip():
            n_blank += 1
            continue
        for chunk in text.split(";"):
            toks = chunk.split()
            assert len(toks) >= 6, f"{image_id}: polygon part with {len(toks) // 2} points"
            assert len(toks) % 2 == 0, f"{image_id}: odd coordinate count"
            v = np.asarray(toks, dtype=np.float64)
            assert np.isfinite(v).all(), f"{image_id}: non-finite coordinate"
            assert (v >= 0).all() and (v <= 1).all(), \
                f"{image_id}: coordinate outside [0, 1] (min {v.min()}, max {v.max()})"
    print(f"submission validated OK ({len(sub) - n_blank} polygons, {n_blank} blank)")


submission = build_submission(test_df.image_id.tolist(), test_probs, GATE,
                              best_threshold, BEST_MIN_AREA, BEST_CLOSE)

validate_submission(submission, sample_sub)

sub_path = os.path.join(CFG.out_dir, "submission.csv")
submission.to_csv(sub_path, index=False)
print(f"\nwrote {sub_path} ({os.path.getsize(sub_path) / 1e6:.1f} MB)")

# Decode what was actually written and confirm it survives the round trip. This is the
# last point where a silent encoding bug can still be caught.
n_poly = int((submission.polygon.str.strip().str.len() > 0).sum())
areas = np.array([polygon_to_mask(t).mean() for t in submission.polygon
                  if t.strip()])
print(f"non-blank rows: {n_poly} of {len(submission)} "
      f"(gate said {int(GATE.sum())})")
if len(areas):
    print(f"predicted mask area fraction: median {np.median(areas):.4f}  "
          f"p5 {np.percentile(areas, 5):.4f}  p95 {np.percentile(areas, 95):.4f}")
    print(f"  OOF ground-truth median for comparison: "
          f"{np.median([g.mean() for g in oof_gt]):.4f}")
    assert areas.min() > 0, "a gated image got an empty mask, guaranteed IoU 0"

## Save probability maps

The submitted result was not this notebook's own submission.csv. It came from blending
these probability maps across several runs with different encoders and seeds, weights
fitted on the pooled OOF. Test maps are written at full 640 so that blend runs on CPU.

In [ ]:
def downsample_stack(stack: np.ndarray, size: int) -> np.ndarray:
    if stack.shape[-1] == size:
        return stack
    return np.stack([cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
                     for m in stack])


# The submitted Level 3 result was a weighted blend of these probability maps across
# several runs of this notebook, with the weights fitted on the pooled OOF. Test maps
# go out at full 640 so a submission can be rebuilt on CPU without any GPU time.
np.savez_compressed(
    os.path.join(CFG.out_dir, "predictions.npz"),
    oof_ids=np.array(oof_ids),
    oof_probs=downsample_stack(oof_probs, CFG.dump_oof_size),
    oof_iou=per_image,
    oof_size=CFG.dump_oof_size,
    test_ids=np.array(test_df.image_id.tolist()),
    test_probs=test_probs if CFG.dump_test_full
    else downsample_stack(test_probs, CFG.dump_oof_size),
    test_size=CFG.eval_size if CFG.dump_test_full else CFG.dump_oof_size,
    gate=GATE,
    threshold=best_threshold,
    min_area_frac=BEST_MIN_AREA,
    close_px=BEST_CLOSE,
)

print(f"wrote predictions.npz | OOF mIoU {OOF_MEAN_IOU:.4f} | threshold "
      f"{best_threshold} | min_area {BEST_MIN_AREA} | close {BEST_CLOSE}")


## What actually moved the score

The per-image IoU distribution above is the thing to read first, since the two failure
modes need opposite fixes:

- Many images at IoU 0.3-0.6 means boundaries are soft. Levers are more epochs (the
  decoder starts from scratch, so check whether per-epoch mIoU was still climbing), a
  heavier dice_weight, and flip TTA, which averages boundaries rather than voting on
  labels and so helps here even though it hurt Levels 1 and 2.
- A tail at IoU < 0.1 means whole regions are missed or hallucinated. Those are
  localisation failures rather than boundary failures, so look at the images before
  touching hyperparameters.

In order of what it was worth:

1. The 5-fold CV ensemble. Averaging five fold models was worth +0.055 public over a
   single holdout model, the largest single gain of the whole project.
2. Blending across encoders and seeds. This run at weight 0.4, an effnetv2_m run at
   0.4 and a second seed at 0.2, fitted on the pooled OOF, took OOF from 0.8417 to
   0.8500 and public to 0.89534.
3. Post-processing, and barely. The threshold and morphology sweep above is worth a
   few thousandths, and an oracle per-image threshold would only be worth 0.013.
4. Not the gate. Level 2's Cat-2 decision costs 10 errors in 8835, so at most ~0.001
   of score, while the same GPU-hour spent on mask quality is worth ~0.033 per +0.10
   of mIoU.

One thing that cost a run: hyperparameters do not transfer across architecture
families. ConvNeXt at EfficientNet's lr=2e-4 over 3 epochs collapsed completely on
Level 2, 514-563 errors per fold. Re-tune the LR before blaming the architecture.